This file performs the following  
Splits the adata into liver and immune collections (Split GSE192740 into immune and liver adata for this)  
For each collection:  
  - Find genes expressed in ≥10 cells per dataset
  - Take intersection across all datasets in collection
  - Concatenate on shared genes (inner join on filtered universe)
  - HVG selection with batch_key='dataset_id', n_top=5000
  - Batch correction (scVI or Harmony) preserving biology  

For each collection perform CellTypist liver model cell annotation  
Check cell counts and create per cell type adata objects  

In [1]:
from pathlib import Path

import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import median_abs_deviation
import numpy as np

# Set Base directory to the location of this script
BASE = Path('c:/Users/ankit/Documents/scFM/train_data/')

# Set working directory to the location of this script 
# data_dir = BASE / 'h5s_common_directory'
data_dir = BASE / 'h5s_common_directory_V2_WIP'
# data_dir = BASE / 'scGPT_data'

# Print contents of the data directory for verification
print(f"Contents of data directory ({data_dir}):")
for item in data_dir.iterdir():
    print(item.name)

Contents of data directory (c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP):
GSE159977.h5ad
GSE174748.h5ad
GSE185477.h5ad
GSE189600.h5ad
GSE190487.h5ad
GSE192740.h5ad
GSE202379.h5ad
GSE212837.h5ad
GSE270488.h5ad


In [2]:
# Read in each h5ad file with the Anndata object variable name being it's file name without the extension and store in dictionary
adata_dict = {}

for h5ad_file in data_dir.glob('*.h5ad'):
    adata_name = h5ad_file.stem  # Get file name without extension
    adata_dict[adata_name] = ad.read_h5ad(h5ad_file)  # Read h5ad file and store in dictionary

c:\Users\ankit\miniconda3\envs\celltypist\Lib\site-packages\anndata\_core\anndata.py:1878: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Sanity Check

In [3]:
for adata_name, adata in adata_dict.items():
    print(f"\nProcessing Anndata object: {adata_name}")
    print(f"Shape of the Anndata object: {adata.shape}")
    print(f"Number of genes: {adata.n_vars}")
    print(f"Number of cells: {adata.n_obs}")


Processing Anndata object: GSE159977
Shape of the Anndata object: (106851, 23993)
Number of genes: 23993
Number of cells: 106851

Processing Anndata object: GSE174748
Shape of the Anndata object: (19038, 27934)
Number of genes: 27934
Number of cells: 19038

Processing Anndata object: GSE185477
Shape of the Anndata object: (125790, 37469)
Number of genes: 37469
Number of cells: 125790

Processing Anndata object: GSE189600
Shape of the Anndata object: (55074, 32277)
Number of genes: 32277
Number of cells: 55074

Processing Anndata object: GSE190487
Shape of the Anndata object: (41995, 19059)
Number of genes: 19059
Number of cells: 41995

Processing Anndata object: GSE192740
Shape of the Anndata object: (69758, 30704)
Number of genes: 30704
Number of cells: 69758

Processing Anndata object: GSE202379
Shape of the Anndata object: (64329, 30596)
Number of genes: 30596
Number of cells: 64329

Processing Anndata object: GSE212837
Shape of the Anndata object: (182302, 33557)
Number of genes: 

In [10]:
print(adata_dict["GSE192740"].obs["Sample name"].value_counts())

# Assign the cells with Sample name "Liver_CD45- Cells_ Human" a assay_type value of scRNA-seq
adata_dict["GSE192740"].obs.loc[adata_dict["GSE192740"].obs["Sample name"] == "Liver_CD45- Cells_ Human", "assay_type"] = "scRNA-seq"

print(adata_dict["GSE192740"].obs["Sample name"].value_counts())


Sample name
Whole Liver Nuclei_Human    39020
Liver_CD45+ Cells_ Human    27830
Liver_CD45- Cells_ Human     2908
Name: count, dtype: int64
Sample name
Whole Liver Nuclei_Human    39020
Liver_CD45+ Cells_ Human    27830
Liver_CD45- Cells_ Human     2908
Name: count, dtype: int64


In [11]:
print(adata_dict["GSE192740"].obs["assay_type"].value_counts())

# Create new GSE192740_liver and GSE192740_immune by filtering GSE192740 for assay_type == 'liver' and assay_type == 'immune' respectively
adata_dict["GSE192740_liver"] = adata_dict["GSE192740"][adata_dict["GSE192740"].obs["assay_type"] == "snRNA-seq"].copy()
adata_dict["GSE192740_immune"] = adata_dict["GSE192740"][adata_dict["GSE192740"].obs["assay_type"] == "scRNA-seq"].copy()

# Print the Sample name value counts for the new GSE192740_liver and GSE192740_immune Anndata objects to verify the filtering
print(adata_dict["GSE192740_liver"].obs["Sample name"].value_counts())
print(adata_dict["GSE192740_immune"].obs["Sample name"].value_counts())

assay_type
snRNA-seq    39020
scRNA-seq    30738
None             0
Name: count, dtype: int64
Sample name
Whole Liver Nuclei_Human    39020
Name: count, dtype: int64
Sample name
Liver_CD45+ Cells_ Human    27830
Liver_CD45- Cells_ Human     2908
Name: count, dtype: int64


In [13]:
# See what is the number of unique genes that are present in all adata objects
unique_genes = set()
for adata_name, adata in adata_dict.items():
    unique_genes.update(adata.var_names)

print(f"Number of unique genes in all adata objects: {len(unique_genes)}")


Number of unique genes in all adata objects: 48422


In [14]:
# See what is the number of genes that are present in all adata objects
common_genes = set(adata_dict["GSE192740"].var_names)
for adata_name, adata in adata_dict.items():
    common_genes.intersection_update(adata.var_names)

print(f"Number of common genes in all adata objects: {len(common_genes)}")

Number of common genes in all adata objects: 12744


In [15]:
# Get rid of the GSE192740 Anndata object from the adata_dict since we have created GSE192740_liver and GSE192740_immune from it
del adata_dict["GSE192740"]



Check for duplicated cells

In [19]:
for name, adata in adata_dict.items():
    print(name)
    print(f"Duplicate cells before: {adata.obs.index.duplicated().sum()}")
    
    # Remove duplicate cells if they exist
    if adata.obs.index.duplicated().any():
        adata = adata[~adata.obs.index.duplicated(keep='first')]
        adata_dict[name] = adata
    
    print(f"Duplicate cells after: {adata.obs.index.duplicated().sum()}")


GSE159977
Duplicate cells before: 3609
Duplicate cells after: 0
GSE174748
Duplicate cells before: 0
Duplicate cells after: 0
GSE185477
Duplicate cells before: 0
Duplicate cells after: 0
GSE189600
Duplicate cells before: 0
Duplicate cells after: 0
GSE190487
Duplicate cells before: 0
Duplicate cells after: 0
GSE202379
Duplicate cells before: 0
Duplicate cells after: 0
GSE212837
Duplicate cells before: 0
Duplicate cells after: 0
GSE270488
Duplicate cells before: 0
Duplicate cells after: 0
GSE192740_liver
Duplicate cells before: 0
Duplicate cells after: 0
GSE192740_immune
Duplicate cells before: 0
Duplicate cells after: 0


Perform annotation

In [2]:
from pathlib import Path

import celltypist
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import median_abs_deviation
import numpy as np
import celltypist
from celltypist import models

c:\Users\ankit\miniconda3\envs\celltypist\Lib\site-packages\celltypist\classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [18]:
model = models.Model.load(model = 'Healthy_Human_Liver.pkl')

In [20]:
for name, adata in adata_dict.items():
    print(name)
    
    # Skip if already annotated
    if any(col.startswith("typist_liver_") for col in adata.obs.columns):
        print("Already annotated with typist_liver_, skipping...")
        print()
        continue
    
    print()
    predictions = celltypist.annotate(adata, model = 'Healthy_Human_Liver.pkl', majority_voting = True)
    adata = predictions.to_adata(prefix="typist_liver_", insert_conf_by="majority_voting")
    adata_dict[name] = adata
    print(adata.obs["typist_liver_majority_voting"].head())
    print(adata.obs["typist_liver_conf_score"].head())

GSE159977



🔬 Input data has 103242 cells and 23993 genes
🔗 Matching reference genes in the model
🧬 2568 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
c:\Users\ankit\miniconda3\envs\celltypist\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
⛓️ Over-clustering input data with resolution set to 25
🗳️ Majority voting the predictions
✅ Majority voting done!
🔬 Input data has 19038 cells and 27934 genes
🔗 Matching reference genes in the model


AAACCCAAGCATATGA-1    Resident NK
AAACCCACAACGGCTC-1        T cells
AAACCCAGTATTCCGA-1    Resident NK
AAACCCAGTTAAGACA-1        T cells
AAACGAAAGTTGTAGA-1        T cells
Name: typist_liver_majority_voting, dtype: category
Categories (11, object): ['B cells', 'Basophils', 'Circulating NK/NKT', 'Macrophages', ..., 'T cells', 'cDC1s', 'cDC2s', 'pDCs']
AAACCCAAGCATATGA-1    0.999979
AAACCCACAACGGCTC-1    0.993928
AAACCCAGTATTCCGA-1    0.999923
AAACCCAGTTAAGACA-1    0.999995
AAACGAAAGTTGTAGA-1    0.998850
Name: typist_liver_conf_score, dtype: float64
GSE174748



🧬 2485 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 10
🗳️ Majority voting the predictions
✅ Majority voting done!
🔬 Input data has 125790 cells and 37469 genes
🔗 Matching reference genes in the model


AAACCCAAGAGAGTGA-1_GSM5325534_healthy1    Macrophages
AAACCCAAGCGGTAGT-1_GSM5325534_healthy1    Hepatocytes
AAACCCACAAGGGTCA-1_GSM5325534_healthy1    Hepatocytes
AAACCCACACAGCATT-1_GSM5325534_healthy1    Hepatocytes
AAACCCACATCATCTT-1_GSM5325534_healthy1    Hepatocytes
Name: typist_liver_majority_voting, dtype: category
Categories (9, object): ['B cells', 'Cholangiocytes', 'Endothelial cells', 'Fibroblasts', ..., 'Macrophages', 'Plasma cells', 'Resident NK', 'T cells']
AAACCCAAGAGAGTGA-1_GSM5325534_healthy1    1.000000
AAACCCAAGCGGTAGT-1_GSM5325534_healthy1    1.000000
AAACCCACAAGGGTCA-1_GSM5325534_healthy1    0.994462
AAACCCACACAGCATT-1_GSM5325534_healthy1    1.000000
AAACCCACATCATCTT-1_GSM5325534_healthy1    1.000000
Name: typist_liver_conf_score, dtype: float64
GSE185477



...

ND_1_ND_AAACGGGTCATCGATG-1    T cells
ND_1_ND_AAAGATGGTGCATCTA-1    T cells
ND_1_ND_AAAGATGTCACAGTAC-1    T cells
ND_1_ND_AAAGATGTCGCGGATC-1    T cells
ND_1_ND_AAAGCAACAGACAAGC-1    T cells
Name: typist_liver_majority_voting, dtype: category
Categories (2, object): ['Resident NK', 'T cells']
ND_1_ND_AAACGGGTCATCGATG-1    0.040775
ND_1_ND_AAAGATGGTGCATCTA-1    0.958100
ND_1_ND_AAAGATGTCACAGTAC-1    0.768228
ND_1_ND_AAAGATGTCGCGGATC-1    0.997641
ND_1_ND_AAAGCAACAGACAAGC-1    0.999992
Name: typist_liver_conf_score, dtype: float64
GSE192740_liver



🧬 2688 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 15
🗳️ Majority voting the predictions
✅ Majority voting done!
🔬 Input data has 30738 cells and 30704 genes
🔗 Matching reference genes in the model


ABU8_AAACCCAAGGCATTTC-1-7          Hepatocytes
ABU8_AAACCCACAACTGTGT-1-7          Hepatocytes
ABU8_AAACCCATCTTAGCCC-1-7    Endothelial cells
ABU8_AAACGAACAGAAATTG-1-7          Hepatocytes
ABU8_AAACGCTAGGATTTGA-1-7          Hepatocytes
Name: typist_liver_majority_voting, dtype: category
Categories (10, object): ['B cells', 'Cholangiocytes', 'Circulating NK/NKT', 'Endothelial cells', ..., 'Macrophages', 'Plasma cells', 'Resident NK', 'T cells']
ABU8_AAACCCAAGGCATTTC-1-7    0.615399
ABU8_AAACCCACAACTGTGT-1-7    1.000000
ABU8_AAACCCATCTTAGCCC-1-7    1.000000
ABU8_AAACGAACAGAAATTG-1-7    1.000000
ABU8_AAACGCTAGGATTTGA-1-7    0.999974
Name: typist_liver_conf_score, dtype: float64
GSE192740_immune



🧬 2688 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 15
🗳️ Majority voting the predictions
✅ Majority voting done!


CS110_AAACCCACACGGTGTC-1-0    Macrophages
CS110_AAACCCACAGTCAGTT-1-0    Resident NK
CS110_AAACCCACATTGCAAC-1-0        T cells
CS110_AAACCCAGTGTGCTTA-1-0        T cells
CS110_AAACCCAGTGTTCATG-1-0        T cells
Name: typist_liver_majority_voting, dtype: category
Categories (16, object): ['B cells', 'Basophils', 'Cholangiocytes', 'Circulating NK/NKT', ..., 'T cells', 'cDC1s', 'cDC2s', 'pDCs']
CS110_AAACCCACACGGTGTC-1-0    1.000000
CS110_AAACCCACAGTCAGTT-1-0    0.999957
CS110_AAACCCACATTGCAAC-1-0    0.999873
CS110_AAACCCAGTGTGCTTA-1-0    0.999996
CS110_AAACCCAGTGTTCATG-1-0    0.998971
Name: typist_liver_conf_score, dtype: float64


In [21]:
# Save each anndata object in the dictionary to a new h5ad file with the same GSE file name in the directory called 'h5s_common_directory_V2_WIP'

output_dir = BASE / 'h5s_common_directory_V2_WIPAnnotated'
output_dir.mkdir(exist_ok=True)  # Create output directory if it doesn't exist
for name, adata in adata_dict.items():
    output_file = output_dir / f"{name}.h5ad"
    adata.write_h5ad(output_file)
    print(f"Saved {name} to {output_file}")

Saved GSE159977 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE159977.h5ad
Saved GSE174748 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE174748.h5ad
Saved GSE185477 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE185477.h5ad
Saved GSE189600 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE189600.h5ad
Saved GSE190487 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE190487.h5ad
Saved GSE202379 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE202379.h5ad
Saved GSE212837 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE212837.h5ad
Saved GSE270488 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE270488.h5ad
Saved GSE192740_liver to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\G

Checkpoint

In [3]:
# Read in each h5ad file with the Anndata object variable name being it's file name without the extension and store in dictionary
output_dir = BASE / 'h5s_common_directory_V2_WIPAnnotated'
adata_dict = {}
for h5ad_file in output_dir.glob('*.h5ad'):
    adata_name = h5ad_file.stem  # Get file name without extension
    adata_dict[adata_name] = ad.read_h5ad(h5ad_file)  # Read h5ad file and store in dictionary

In [39]:
output_dir = BASE / 'h5s_common_directory_V2_WIPAnnotated'
immune_adata_dict = {}
immune_collection = ["GSE190487", "GSE159977", "GSE192740_immune"]
for h5ad_file in output_dir.glob('*.h5ad'):
    adata_name = h5ad_file.stem  # Get file name without extension
    if any(immune_name in adata_name for immune_name in immune_collection):
        immune_adata_dict[adata_name] = ad.read_h5ad(h5ad_file)  # Read h5ad file and store in dictionary

In [41]:
from pathlib import Path

import celltypist
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import median_abs_deviation
import numpy as np
import celltypist
from celltypist import models

In [43]:
model = models.Model.load(model = 'Immune_All_Low.pkl')

In [45]:
for name, adata in immune_adata_dict.items():
    print(name)
    
    # Skip if already annotated
    if any(col.startswith("typist_immune_") for col in adata.obs.columns):
        print("Already annotated with typist_immune_, skipping...")
        print()
        continue
    
    print()
    predictions = celltypist.annotate(adata, model = 'Immune_All_Low.pkl', majority_voting = True)
    adata = predictions.to_adata(prefix="typist_immune_", insert_conf_by="majority_voting")
    immune_adata_dict[name] = adata
    print(adata.obs["typist_immune_majority_voting"].head())
    print(adata.obs["typist_immune_conf_score"].head())

🔬 Input data has 41995 cells and 19059 genes
🔗 Matching reference genes in the model


GSE159977
Already annotated with typist_immune_, skipping...

GSE190487



🧬 5531 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Detected a neighborhood graph in the input object, will run over-clustering on the basis of it
⛓️ Over-clustering input data with resolution set to 20
🗳️ Majority voting the predictions
✅ Majority voting done!
🔬 Input data has 30738 cells and 30704 genes
🔗 Matching reference genes in the model


barcode
AAACCCAAGACGAAGA-1    Memory B cells
AAACCCAAGACTCTAC-1    Memory B cells
AAACCCAAGAGAGGTA-1        Myelocytes
AAACCCAAGAGCAGAA-1    Memory B cells
AAACCCAAGATGTAGT-1    Memory B cells
Name: typist_immune_majority_voting, dtype: category
Categories (6, object): ['Memory B cells', 'Myelocytes', 'NK cells', 'Naive B cells', 'Plasma cells', 'Tcm/Naive helper T cells']
barcode
AAACCCAAGACGAAGA-1    0.003851
AAACCCAAGACTCTAC-1    0.124588
AAACCCAAGAGAGGTA-1    0.048412
AAACCCAAGAGCAGAA-1    0.002876
AAACCCAAGATGTAGT-1    0.441437
Name: typist_immune_conf_score, dtype: float64
GSE192740_immune



🧬 5263 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Detected a neighborhood graph in the input object, will run over-clustering on the basis of it
⛓️ Over-clustering input data with resolution set to 15
🗳️ Majority voting the predictions
✅ Majority voting done!


CS110_AAACCCACACGGTGTC-1-0    Erythrophagocytic macrophages
CS110_AAACCCACAGTCAGTT-1-0                   CD16- NK cells
CS110_AAACCCACATTGCAAC-1-0                       MAIT cells
CS110_AAACCCAGTGTGCTTA-1-0                       MAIT cells
CS110_AAACCCAGTGTTCATG-1-0        Tem/Trm cytotoxic T cells
Name: typist_immune_majority_voting, dtype: category
Categories (25, object): ['B cells', 'CD16+ NK cells', 'CD16- NK cells', 'Classical monocytes', ..., 'Tem/Temra cytotoxic T cells', 'Tem/Trm cytotoxic T cells', 'Trm cytotoxic T cells', 'pDC']
CS110_AAACCCACACGGTGTC-1-0    0.892503
CS110_AAACCCACAGTCAGTT-1-0    0.999591
CS110_AAACCCACATTGCAAC-1-0    0.978611
CS110_AAACCCAGTGTGCTTA-1-0    0.999969
CS110_AAACCCAGTGTTCATG-1-0    0.205449
Name: typist_immune_conf_score, dtype: float64


In [46]:
# 194 351 cells total before filtering for confidence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

score_col = "typist_immune_conf_score"

# Choose thresholds from 50% to 100%
# Change step to 0.01 if you want finer thresholds
thresholds = np.round(np.arange(0.50, 1.001, 0.05), 2)

# Collect confidence scores from all adata objects
rows = []

for gse_name, adata in immune_adata_dict.items():
    if score_col not in adata.obs.columns:
        print(f"WARNING: {gse_name} does not have {score_col}. Skipping.")
        continue

    scores = pd.to_numeric(adata.obs[score_col], errors="coerce")

    temp = pd.DataFrame({
        "GSE": gse_name,
        "conf_score": scores.values
    })

    rows.append(temp)

immune_conf_df = pd.concat(rows, ignore_index=True)

print(f"Total cells across included objects: {immune_conf_df.shape[0]:,}")
print(f"Cells with non-missing {score_col}: {immune_conf_df['conf_score'].notna().sum():,}")
print(f"Cells with missing {score_col}: {immune_conf_df['conf_score'].isna().sum():,}")

# Overall threshold impact
overall_threshold_summary = []

total_cells = immune_conf_df.shape[0]

for thresh in thresholds:
    kept = (immune_conf_df["conf_score"] >= thresh).sum()
    removed = total_cells - kept

    overall_threshold_summary.append({
        "threshold": thresh,
        "cells_kept": kept,
        "cells_removed": removed,
        "percent_kept": kept / total_cells * 100,
        "percent_removed": removed / total_cells * 100
    })

overall_threshold_summary = pd.DataFrame(overall_threshold_summary)

display(overall_threshold_summary)

Total cells across included objects: 175,975
Cells with non-missing typist_immune_conf_score: 175,975
Cells with missing typist_immune_conf_score: 0


,threshold,cells_kept,cells_removed,percent_kept,percent_removed
0,0.50,87736,88239,49.857082,50.142918
1,0.55,85631,90344,48.660889,51.339111
2,0.60,83487,92488,47.442534,52.557466
3,0.65,81297,94678,46.198039,53.801961
4,0.70,79143,96832,44.974002,55.025998
5,0.75,76891,99084,43.694275,56.305725
6,0.80,74360,101615,42.256002,57.743998
7,0.85,71441,104534,40.597244,59.402756
8,0.90,67720,108255,38.482739,61.517261
9,0.95,62076,113899,35.275465,64.724535


In [49]:
import gc
import pandas as pd

score_col = "typist_immune_conf_score"
threshold = 0.50

filter_summary = []

for gse_name in list(immune_adata_dict.keys()):
    adata = immune_adata_dict[gse_name]

    print("=" * 80)
    print(f"Processing: {gse_name}")
    print(f"Before filtering: {adata.shape}")

    if score_col not in adata.obs.columns:
        print(f"WARNING: {score_col} not found. Leaving {gse_name} unchanged.")
        filter_summary.append({
            "GSE": gse_name,
            "before_cells": adata.n_obs,
            "after_cells": adata.n_obs,
            "removed_cells": 0,
            "percent_kept": 100.0,
            "status": f"Skipped: missing {score_col}"
        })
        continue

    scores = pd.to_numeric(adata.obs[score_col], errors="coerce")
    keep_mask = scores >= threshold

    before_n = adata.n_obs
    after_n = int(keep_mask.sum())
    removed_n = before_n - after_n

    immune_adata_dict[gse_name] = adata[keep_mask.values].copy()

    print(f"After filtering: {immune_adata_dict[gse_name].shape}")
    print(f"Removed cells: {removed_n:,}")
    print(f"Kept cells: {after_n:,} ({after_n / before_n * 100:.2f}%)")

    filter_summary.append({
        "GSE": gse_name,
        "before_cells": before_n,
        "after_cells": after_n,
        "removed_cells": removed_n,
        "percent_kept": after_n / before_n * 100,
        "status": "Filtered"
    })

    del adata
    gc.collect()

filter_summary = pd.DataFrame(filter_summary)
display(filter_summary)

print("=" * 80)
print(f"Total cells before: {filter_summary['before_cells'].sum():,}")
print(f"Total cells after: {filter_summary['after_cells'].sum():,}")
print(f"Total cells removed: {filter_summary['removed_cells'].sum():,}")
print(f"Overall percent kept: {filter_summary['after_cells'].sum() / filter_summary['before_cells'].sum() * 100:.2f}%")

Processing: GSE159977
Before filtering: (103242, 23993)
After filtering: (60513, 23993)
Removed cells: 42,729
Kept cells: 60,513 (58.61%)
Processing: GSE190487
Before filtering: (41995, 19059)
After filtering: (5161, 19059)
Removed cells: 36,834
Kept cells: 5,161 (12.29%)
Processing: GSE192740_immune
Before filtering: (30738, 30704)
After filtering: (22062, 30704)
Removed cells: 8,676
Kept cells: 22,062 (71.77%)


,GSE,before_cells,after_cells,removed_cells,percent_kept,status
0,GSE159977,103242,60513,42729,58.612774,Filtered
1,GSE190487,41995,5161,36834,12.289558,Filtered
2,GSE192740_immune,30738,22062,8676,71.774351,Filtered


Total cells before: 175,975
Total cells after: 87,736
Total cells removed: 88,239
Overall percent kept: 49.86%


In [5]:
import gc
import pandas as pd

score_col = "typist_liver_conf_score"
threshold = 0.50

filter_summary = []

for gse_name in list(adata_dict.keys()):
    adata = adata_dict[gse_name]

    print("=" * 80)
    print(f"Processing: {gse_name}")
    print(f"Before filtering: {adata.shape}")

    if score_col not in adata.obs.columns:
        print(f"WARNING: {score_col} not found. Leaving {gse_name} unchanged.")
        filter_summary.append({
            "GSE": gse_name,
            "before_cells": adata.n_obs,
            "after_cells": adata.n_obs,
            "removed_cells": 0,
            "percent_kept": 100.0,
            "status": f"Skipped: missing {score_col}"
        })
        continue

    scores = pd.to_numeric(adata.obs[score_col], errors="coerce")
    keep_mask = scores >= threshold

    before_n = adata.n_obs
    after_n = int(keep_mask.sum())
    removed_n = before_n - after_n

    adata_dict[gse_name] = adata[keep_mask.values].copy()

    print(f"After filtering: {adata_dict[gse_name].shape}")
    print(f"Removed cells: {removed_n:,}")
    print(f"Kept cells: {after_n:,} ({after_n / before_n * 100:.2f}%)")

    filter_summary.append({
        "GSE": gse_name,
        "before_cells": before_n,
        "after_cells": after_n,
        "removed_cells": removed_n,
        "percent_kept": after_n / before_n * 100,
        "status": "Filtered"
    })

    del adata
    gc.collect()

filter_summary = pd.DataFrame(filter_summary)
display(filter_summary)

print("=" * 80)
print(f"Total cells before: {filter_summary['before_cells'].sum():,}")
print(f"Total cells after: {filter_summary['after_cells'].sum():,}")
print(f"Total cells removed: {filter_summary['removed_cells'].sum():,}")
print(f"Overall percent kept: {filter_summary['after_cells'].sum() / filter_summary['before_cells'].sum() * 100:.2f}%")

Processing: GSE159977
Before filtering: (103242, 23993)
After filtering: (89968, 23993)
Removed cells: 13,274
Kept cells: 89,968 (87.14%)
Processing: GSE174748
Before filtering: (19038, 27934)
After filtering: (15598, 27934)
Removed cells: 3,440
Kept cells: 15,598 (81.93%)
Processing: GSE185477
Before filtering: (125790, 37469)
After filtering: (100474, 37469)
Removed cells: 25,316
Kept cells: 100,474 (79.87%)
Processing: GSE189600
Before filtering: (55074, 32277)
After filtering: (50813, 32277)
Removed cells: 4,261
Kept cells: 50,813 (92.26%)
Processing: GSE190487
Before filtering: (41995, 19059)
After filtering: (30502, 19059)
Removed cells: 11,493
Kept cells: 30,502 (72.63%)
Processing: GSE192740_immune
Before filtering: (30738, 30704)
After filtering: (29274, 30704)
Removed cells: 1,464
Kept cells: 29,274 (95.24%)
Processing: GSE192740_liver
Before filtering: (39020, 30704)
After filtering: (30534, 30704)
Removed cells: 8,486
Kept cells: 30,534 (78.25%)
Processing: GSE202379
Before

,GSE,before_cells,after_cells,removed_cells,percent_kept,status
0,GSE159977,103242,89968,13274,87.142829,Filtered
1,GSE174748,19038,15598,3440,81.930875,Filtered
2,GSE185477,125790,100474,25316,79.874394,Filtered
3,GSE189600,55074,50813,4261,92.263137,Filtered
4,GSE190487,41995,30502,11493,72.632456,Filtered
5,GSE192740_immune,30738,29274,1464,95.237166,Filtered
6,GSE192740_liver,39020,30534,8486,78.252178,Filtered
7,GSE202379,64329,61947,2382,96.297160,Filtered
8,GSE212837,182302,171806,10496,94.242521,Filtered
9,GSE270488,18376,17583,793,95.684589,Filtered


Total cells before: 679,904
Total cells after: 598,499
Total cells removed: 81,405
Overall percent kept: 88.03%


In [6]:
# Liver  GSEs

liver_collection = ["GSE212837", "GSE189600", "GSE174748", "GSE192740_liver", "GSE185477", "GSE202379"]
immune_collection = ["GSE190487", "GSE159977", "GSE192740_immune"]
external_validation = ["GSE270488"]

liver_adata_dict = {}
immune_adata_dict = {}

for gse_name in liver_collection:
    if gse_name in adata_dict:
        liver_adata_dict[gse_name] = adata_dict[gse_name]
    else:
        print(f"WARNING: {gse_name} not found in adata_dict.")

for gse_name in immune_collection:
    if gse_name in adata_dict:
        immune_adata_dict[gse_name] = adata_dict[gse_name]
    else:
        print(f"WARNING: {gse_name} not found in adata_dict.")

for gse_name in external_validation:
    if gse_name in adata_dict:
        external_validation_adata = adata_dict[gse_name]
    else:
        print(f"WARNING: {gse_name} not found in adata_dict.")



In [7]:
# Remove adata_dict to free up memory
del adata_dict
gc.collect()

7

In [37]:
# Create a table of rows as different values of typist_liver_majority_voting and columns as diferent GSEs with values as the number of cells that have that typist_liver_majority_voting value for the immune GSEs
# First create an empty dataframe with index as the unique values of typist_liver_majority_voting across all immune GSEs and columns as the immune GSE names
immune_cell_types = sorted(set().union(*(adata.obs["typist_liver_majority_voting"].unique() for adata in immune_adata_dict.values())))
immune_cell_table = pd.DataFrame(
    {gse: adata.obs["typist_liver_majority_voting"].value_counts() for gse, adata in immune_adata_dict.items()}
)

# Add a total count column and sort by that
immune_cell_table["total_counts"] = immune_cell_table.sum(axis=1)
immune_cell_table = immune_cell_table.sort_values("total_counts", ascending=False)

print(immune_cell_table)

                              GSE190487  GSE159977  GSE192740_immune  \
typist_liver_majority_voting                                           
T cells                         30502.0    55340.0             15720   
Resident NK                         NaN    15777.0              2625   
Circulating NK/NKT                  NaN    10949.0              2515   
Mono+mono derived cells             NaN     2783.0              1951   
B cells                             NaN     4078.0               368   
Macrophages                         NaN       41.0              2505   
Endothelial cells                   NaN        NaN              2217   
pDCs                                NaN      398.0               208   
cDC2s                               NaN      254.0               164   
Neutrophils                         NaN        NaN               416   
Plasma cells                        NaN       76.0               263   
Basophils                           NaN      166.0              

In [50]:
# Create a table of rows as different values of typist_immune_majority_voting and columns as diferent GSEs with values as the number of cells that have that typist_immune_majority_voting value for the immune GSEs
# First create an empty dataframe with index as the unique values of typist_immune_majority_voting across all immune GSEs and columns as the immune GSE names
immune_cell_types = sorted(set().union(*(adata.obs["typist_immune_majority_voting"].unique() for adata in immune_adata_dict.values())))
immune_cell_table = pd.DataFrame(
    {gse: adata.obs["typist_immune_majority_voting"].value_counts() for gse, adata in immune_adata_dict.items()}
)

# Add a total count column and sort by that
immune_cell_table["total_counts"] = immune_cell_table.sum(axis=1)
immune_cell_table = immune_cell_table.sort_values("total_counts", ascending=False)

print(immune_cell_table)

                               GSE159977  GSE190487  GSE192740_immune  \
typist_immune_majority_voting                                           
CD16- NK cells                   16542.0        NaN            2744.0   
CD16+ NK cells                   10665.0        NaN            2462.0   
Tem/Trm cytotoxic T cells         7675.0        NaN            3549.0   
MAIT cells                        5388.0        NaN            4714.0   
Memory B cells                     699.0     4943.0               NaN   
Tem/Temra cytotoxic T cells       3643.0        NaN            1571.0   
Tem/Effector helper T cells       4467.0        NaN             364.0   
Naive B cells                     2515.0        1.0               NaN   
Classical monocytes                915.0        NaN            1123.0   
NK cells                          1926.0        1.0               NaN   
Non-classical monocytes           1354.0        NaN             422.0   
Endothelial cells                    NaN        NaN

In [12]:
# Create a table of rows as different values of typist_liver_majority_voting and columns as diferent GSEs with values as the number of cells that have that typist_liver_majority_voting value for that GSE
cell_types = sorted(set().union(*(counts.index for counts in liver_cell_counts.values())))
liver_cell_table = pd.DataFrame(
    {gse_name: counts.reindex(cell_types).fillna(0).astype(int)
     for gse_name, counts in liver_cell_counts.items()},
    index=cell_types,
)
print(liver_cell_table)

                         GSE212837  GSE189600  GSE174748  GSE192740_liver  \
B cells                          0          0         19               22   
Basophils                       52          0          0                0   
Cholangiocytes                5079       3394        451              790   
Circulating NK/NKT              28          0          0               48   
Endothelial cells            20344       2400       1418             3361   
Fibroblasts                   5974       3814        759             2747   
Hepatocytes                 132715      41181      11955            22179   
Macrophages                   5666         24        479             1082   
Mono+mono derived cells         20          0          0                0   
Plasma cells                   124          0         49               35   
Resident NK                    344          0         98               63   
T cells                       1460          0        370              207   

In [16]:
# Find the total counts per cell type across all GSEs and sort by that
liver_cell_table["total_counts"] = liver_cell_table.sum(axis=1)
liver_cell_table = liver_cell_table.sort_values("total_counts", ascending=False)
print(liver_cell_table)

                         GSE212837  GSE189600  GSE174748  GSE192740_liver  \
Hepatocytes                 132715      41181      11955            22179   
Endothelial cells            20344       2400       1418             3361   
Fibroblasts                   5974       3814        759             2747   
Cholangiocytes                5079       3394        451              790   
Macrophages                   5666         24        479             1082   
T cells                       1460          0        370              207   
Resident NK                    344          0         98               63   
Circulating NK/NKT              28          0          0               48   
Plasma cells                   124          0         49               35   
B cells                          0          0         19               22   
Mono+mono derived cells         20          0          0                0   
Basophils                       52          0          0                0   

In [ ]:
cell_types_to_adata = ["Hepatocytes", "Endothelial cells", "Fibroblasts", "Cholangiocytes", "Macrophages", "T cells", "Resident NK"]
output_dir = BASE / 'liver_cell_type_adata'
output_dir.mkdir(exist_ok=True)

cell_type_adata_dict = {}

for cell_type in cell_types_to_adata:
    subsets = []
    gse_keys = []

    for gse_name, adata in liver_adata_dict.items():
        if "typist_liver_majority_voting" not in adata.obs.columns:
            print(f"WARNING: {gse_name} does not have typist_liver_majority_voting column. Skipping.")
            continue

        subset = adata[adata.obs["typist_liver_majority_voting"] == cell_type].copy()
        if subset.n_obs == 0:
            continue

        subsets.append(subset)
        gse_keys.append(gse_name)

    if not subsets:
        print(f"No cells found for {cell_type}, skipping.")
        continue

    cell_type_adata = ad.concat(subsets, join="outer", label="GSE", keys=gse_keys)
    cell_type_adata_dict[cell_type] = cell_type_adata
    print(f"Stored {cell_type} adata with {cell_type_adata.n_obs} cells from {len(subsets)} GSEs")


In [51]:
cell_types_to_adata = [ 
    "CD16- NK cells",
    "CD16+ NK cells",
    "Tem/Trm cytotoxic T cells",
    "MAIT cells",
    "Memory B cells",
    "Tem/Temra cytotoxic T cells",
    "Tem/Effector helper T cells",
    "Naive B cells",
    "Classical monocytes",
    "NK cells",
    "Non-classical monocytes",
    "Endothelial cells",
    "Tcm/Naive helper T cells",
    "Kupffer cells",
    "CRTAM+ gamma-delta T cells",
    "pDC",
]


output_dir = BASE / 'immune_cell_type_adata'
output_dir.mkdir(exist_ok=True)

cell_type_adata_dict = {}

for cell_type in cell_types_to_adata:
    subsets = []
    gse_keys = []

    for gse_name, adata in immune_adata_dict.items():
        if "typist_immune_majority_voting" not in adata.obs.columns:
            print(f"WARNING: {gse_name} does not have typist_immune_majority_voting column. Skipping.")
            continue

        subset = adata[adata.obs["typist_immune_majority_voting"] == cell_type].copy()
        if subset.n_obs == 0:
            continue

        subsets.append(subset)
        gse_keys.append(gse_name)

    if not subsets:
        print(f"No cells found for {cell_type}, skipping.")
        continue

    cell_type_adata = ad.concat(subsets, join="outer", label="GSE", keys=gse_keys)
    cell_type_adata_dict[cell_type] = cell_type_adata
    print(f"Stored {cell_type} adata with {cell_type_adata.n_obs} cells from {len(subsets)} GSEs")


Stored CD16- NK cells adata with 19286 cells from 2 GSEs
Stored CD16+ NK cells adata with 13127 cells from 2 GSEs
Stored Tem/Trm cytotoxic T cells adata with 11224 cells from 2 GSEs
Stored MAIT cells adata with 10102 cells from 2 GSEs
Stored Memory B cells adata with 5642 cells from 2 GSEs
Stored Tem/Temra cytotoxic T cells adata with 5214 cells from 2 GSEs
Stored Tem/Effector helper T cells adata with 4831 cells from 2 GSEs
Stored Naive B cells adata with 2516 cells from 2 GSEs
Stored Classical monocytes adata with 2038 cells from 2 GSEs
Stored NK cells adata with 1927 cells from 2 GSEs
Stored Non-classical monocytes adata with 1776 cells from 2 GSEs
Stored Endothelial cells adata with 1750 cells from 1 GSEs
Stored Tcm/Naive helper T cells adata with 1666 cells from 3 GSEs
Stored Kupffer cells adata with 1414 cells from 1 GSEs
Stored CRTAM+ gamma-delta T cells adata with 1019 cells from 1 GSEs
Stored pDC adata with 539 cells from 2 GSEs


In [52]:
for adata_name, adata in cell_type_adata_dict.items():
    print(adata_name)
    print(f"Shape: {adata.shape}")
    print("==============================")

print(cell_type_adata_dict["Hepatocytes"].obs["GSE"].value_counts())

for adata_name, adata in liver_adata_dict.items():
    print(adata_name)
    print(f"Shape: {adata.shape}")
    print("==============================")

CD16- NK cells
Shape: (19286, 32834)
CD16+ NK cells
Shape: (13127, 32834)
Tem/Trm cytotoxic T cells
Shape: (11224, 32834)
MAIT cells
Shape: (10102, 32834)
Memory B cells
Shape: (5642, 27749)
Tem/Temra cytotoxic T cells
Shape: (5214, 32834)
Tem/Effector helper T cells
Shape: (4831, 32834)
Naive B cells
Shape: (2516, 27749)
Classical monocytes
Shape: (2038, 32834)
NK cells
Shape: (1927, 27749)
Non-classical monocytes
Shape: (1776, 32834)
Endothelial cells
Shape: (1750, 30704)
Tcm/Naive helper T cells
Shape: (1666, 36390)
Kupffer cells
Shape: (1414, 30704)
CRTAM+ gamma-delta T cells
Shape: (1019, 23993)
pDC
Shape: (539, 32834)


KeyError: 'Hepatocytes'

In [29]:
# Subset the gene var_names for each cell type specific adata object to only include the common genes across all adata objects
for cell_type, adata in cell_type_adata_dict.items():
    common_genes_in_adata = set(adata.var_names).intersection(common_genes)
    cell_type_adata_dict[cell_type] = adata[:, list(common_genes_in_adata)].copy()
    print(f"{cell_type}: {len(common_genes_in_adata)} common genes retained")

Hepatocytes: 17193 common genes retained
Endothelial cells: 17193 common genes retained
Fibroblasts: 17193 common genes retained
Cholangiocytes: 17193 common genes retained
Macrophages: 17193 common genes retained
T cells: 17193 common genes retained
Resident NK: 17193 common genes retained


In [53]:
# Subset the gene var_names for each cell type specific adata object to only include the common genes across all adata objects
for cell_type, adata in cell_type_adata_dict.items():
    common_genes_in_adata = set(adata.var_names).intersection(common_genes)
    cell_type_adata_dict[cell_type] = adata[:, list(common_genes_in_adata)].copy()
    print(f"{cell_type}: {len(common_genes_in_adata)} common genes retained")

CD16- NK cells: 17193 common genes retained
CD16+ NK cells: 17193 common genes retained
Tem/Trm cytotoxic T cells: 17193 common genes retained
MAIT cells: 17193 common genes retained
Memory B cells: 15528 common genes retained
Tem/Temra cytotoxic T cells: 17193 common genes retained
Tem/Effector helper T cells: 17193 common genes retained
Naive B cells: 15528 common genes retained
Classical monocytes: 17193 common genes retained
NK cells: 15528 common genes retained
Non-classical monocytes: 17193 common genes retained
Endothelial cells: 17193 common genes retained
Tcm/Naive helper T cells: 17193 common genes retained
Kupffer cells: 17193 common genes retained
CRTAM+ gamma-delta T cells: 15359 common genes retained
pDC: 17193 common genes retained


In [54]:
# View a sample of gene var names for one of the cell type specific adata objects
print(cell_type_adata_dict["CD16- NK cells"].var_names[:10])

# Print the number of unique genes in the CD16- NK cells adata object to a csv file
cd16_nk_genes = set(cell_type_adata_dict["CD16- NK cells"].var_names)
with open(output_dir / "cd16_nk_genes.csv", "w") as f:
    f.write("gene\n")
    for gene in sorted(cd16_nk_genes):
        f.write(f"{gene}\n")
print(output_dir)

Index(['OSCP1', 'RAPGEF4', 'STT3A', 'SWT1', 'ENG', 'DND1', 'LINC01054',
       'GPR55', 'SKA2', 'ART5'],
      dtype='object')
c:\Users\ankit\Documents\scFM\train_data\immune_cell_type_adata


In [25]:
# View a sample of gene var names for one of the cell type specific adata objects
print(cell_type_adata_dict["Hepatocytes"].var_names[:10])

# Print the number of unique genes in the Hepatocytes adata object to a csv file
hepatocyte_genes = set(cell_type_adata_dict["Hepatocytes"].var_names)
with open(output_dir / "hepatocyte_genes.csv", "w") as f:
    f.write("gene\n")
    for gene in sorted(hepatocyte_genes):
        f.write(f"{gene}\n")
print(output_dir)

Index(['AL627309.1', 'FAM87B', 'LINC00115', 'FAM41C', 'AL645608.2', 'SAMD11',
       'NOC2L', 'KLHL17', 'PLEKHN1', 'HES4'],
      dtype='object')
c:\Users\ankit\Documents\scFM\train_data\liver_cell_type_adata


In [13]:
# See what is the number of unique genes that are present in all adata objects
unique_genes = set()
for adata_name, adata in liver_adata_dict.items():
    unique_genes.update(adata.var_names)

print(f"Number of unique genes in all adata objects: {len(unique_genes)}")

Number of unique genes in all adata objects: 48098


In [15]:
# See what is the number of genes that are present in all adata objects
common_genes = set(liver_adata_dict["GSE192740_liver"].var_names)
for adata_name, adata in liver_adata_dict.items():
    common_genes.intersection_update(adata.var_names)

print(f"Number of common genes in all adata objects: {len(common_genes)}")

Number of common genes in all adata objects: 17193


In [57]:
# Save the cell type specific adata objects to h5ad files in the output directory with file names as the cell type name
output_dir = BASE / 'immune_cell_type_adata'
for cell_type, adata in cell_type_adata_dict.items():
    # If the file name has any / or \ in the cell type name, replace them with - to avoid issues with file paths
    safe_cell_type = cell_type.replace("/", "-").replace("\\", "-")
    output_file = output_dir / f"{safe_cell_type}.h5ad"
    adata.write_h5ad(output_file)
    print(f"Saved {cell_type} adata to {output_file}")

Saved CD16- NK cells adata to c:\Users\ankit\Documents\scFM\train_data\immune_cell_type_adata\CD16- NK cells.h5ad
Saved CD16+ NK cells adata to c:\Users\ankit\Documents\scFM\train_data\immune_cell_type_adata\CD16+ NK cells.h5ad
Saved Tem/Trm cytotoxic T cells adata to c:\Users\ankit\Documents\scFM\train_data\immune_cell_type_adata\Tem-Trm cytotoxic T cells.h5ad
Saved MAIT cells adata to c:\Users\ankit\Documents\scFM\train_data\immune_cell_type_adata\MAIT cells.h5ad
Saved Memory B cells adata to c:\Users\ankit\Documents\scFM\train_data\immune_cell_type_adata\Memory B cells.h5ad
Saved Tem/Temra cytotoxic T cells adata to c:\Users\ankit\Documents\scFM\train_data\immune_cell_type_adata\Tem-Temra cytotoxic T cells.h5ad
Saved Tem/Effector helper T cells adata to c:\Users\ankit\Documents\scFM\train_data\immune_cell_type_adata\Tem-Effector helper T cells.h5ad
Saved Naive B cells adata to c:\Users\ankit\Documents\scFM\train_data\immune_cell_type_adata\Naive B cells.h5ad
Saved Classical monocyte